In [1]:
pip install PyMuPDF



[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tiktoken


[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install openpyxl


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
# 📌 Imports
import json
import fitz  # PyMuPDF
import openai
import boto3
import io
import requests
import tiktoken
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from reportlab.lib.units import cm
from reportlab.lib.utils import ImageReader
from reportlab.lib.enums import TA_CENTER
from openai import OpenAI
import os
from dotenv import load_dotenv
from openpyxl import Workbook, load_workbook
import datetime
from botocore.exceptions import ClientError
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

load_dotenv()

client = OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))


class Processor:
    def __init__(self):
        # S3 client used across methods
        self.S3_CLIENT = boto3.client(
            "s3",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name="us-east-1"
        )
        # DynamoDB resource and table references
        self.DDB = boto3.resource(
            "dynamodb",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name=os.getenv("AWS_REGION", "us-east-1")
        )
        # Name of the students table
        self.STUDENTS_TABLE = self.DDB.Table(os.getenv("STUDENTS_TABLE_NAME", "Students"))
        # Name of class table (for later use - not used here but keep for future)
        self.CLASS_TABLE = self.DDB.Table(os.getenv("CLASS_TABLE_NAME", "Class"))

        self.S3_BUCKET_NAME = os.getenv("EVALUATED_REPORTS_BUCKET", "evaluated-reports")
        # bucket to store excel trackers
        self.EXCEL_BUCKET = os.getenv("ASSIGNMENT_EXCEL_BUCKET", "assignment-excels")
        # bucket where assignment master files are stored
        self.GENERATED_ASSIGNMENTS_BUCKET = os.getenv("GENERATED_ASSIGNMENTS_BUCKET", "generated-assignments")

    # ---------- PDF extraction ----------
    def read_pdf(self, file_bytes):
        text = ""
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            text += page.get_text() + "\n"
        return text

    # ---------- GPT output cleaner ----------
    def cleaner(self, output={}):
        try:
            raw_text = output['evaluation'].strip()
            if raw_text.startswith("```json"):
                raw_text = raw_text[7:]
            if raw_text.endswith("```"):
                raw_text = raw_text[:-3]
            cleaned_json = json.loads(raw_text)
            return cleaned_json
        except Exception as e:
            print(f"JSON Parsing Failed: {e}")
            return {"error": "Failed to parse GPT output", "raw": output}

    # ---------- GPT evaluation ----------
    def evaluate_entire_assignment(self, content):
        prompt = f"""
        You are a strict assignment evaluator who excels in every known technology.

        The student has submitted an assignment containing multiple questions and answers . Please:
        1. Identify all distinct questions and their answers.
        2. For each, return feedback, a score from 0 to 10, and whether it's AI-generated or not.

        Respond **only in this JSON format**:
        {{
        "evaluation": [
            {{
            "Q1": {{
                "Question1": "...",
                "feedback": "...",
                "score": 0-10,
                "verdict": "AI-generated" or "Not AI-generated"
            }}
            }},
            ...
        ]
        }}

        Input:
        \"\"\"{content}\"\"\"
        """
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a Python assignment evaluator."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=4096
            )
            result = response.model_dump()
            return result["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"OpenAI Evaluation Failed: {e}")
            return json.dumps({"error": "Evaluation failed", "details": str(e)})

    # ---------- chunking & tokens ----------
    def chunk_text(self, text, max_words=3000):
        words = text.split()
        chunks = []
        for i in range(0, len(words), max_words):
            chunks.append(" ".join(words[i:i+max_words]))
        return chunks

    def count_tokens(self, text, model="gpt-4o"):
        encoding = tiktoken.encoding_for_model(model)
        return len(encoding.encode(text))

    # ---------- verification ----------
    def verify_evaluation(self, eval_text):
        MAX_TOKENS = 100000
        token_count = self.count_tokens(eval_text)

        if token_count > MAX_TOKENS:
            eval_text = eval_text[:50000]  # Naive trim

        prompt = f"""
        You are a verification expert.
        Does this JSON include question, feedback, score (0–10), and verdict (AI or Not AI)?
        Return True if valid, else False and why.

        {eval_text}
        """
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You're a verification expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=1200
        )
        result = response.model_dump()
        # result may have choices; safely extract
        try:
            verdict = result["choices"][0]["message"]["content"].strip()
        except Exception:
            verdict = ""
        return (True, "") if verdict.lower().startswith("true") else (False, verdict or result)

    # ---------- sorting ----------
    def sort_evaluation(self, evaluation_list):
        def extract_q_num(item):
            key = list(item.keys())[0]
            # remove non-numeric characters and convert
            try:
                return int(''.join(filter(str.isdigit, key)))
            except Exception:
                return 0
        return sorted(evaluation_list, key=extract_q_num)

    # ---------- PDF decorator & generation ----------
    def decorator(self, canvas_obj, doc):
        width, height = A4
        logo_path = "image.png"
        try:
            logo = ImageReader(logo_path)
            logo_width, logo_height = 200, 200
            x = (width - logo_width) / 2
            y = (height - logo_height) / 2

            canvas_obj.saveState()
            try:
                canvas_obj.setFillAlpha(0.1)
            except AttributeError:
                pass
            canvas_obj.drawImage(logo, x, y, width=logo_width, height=logo_height, mask='auto')
            canvas_obj.restoreState()
        except Exception as e:
            # If logo not found or fails, continue without it
            print(f"Logo draw skipped: {e}")

        canvas_obj.setStrokeColor(colors.grey)
        canvas_obj.setLineWidth(2)
        canvas_obj.rect(10, 10, width - 20, height - 20)

    def json_to_pdf_report(self, data, student_name, assignment_name, output_path="report.pdf"):
        pdfmetrics.registerFont(TTFont('DejaVuSans', 'dejavu-sans/DejaVuSans.ttf'))
        pdfmetrics.registerFont(TTFont('DejaVuSans-Bold', 'dejavu-sans/DejaVuSans-Bold.ttf'))
        pdfmetrics.registerFontFamily('DejaVuSans',
                                     normal='DejaVuSans',
                                     bold='DejaVuSans-Bold')

        doc = SimpleDocTemplate(output_path, pagesize=A4,
                                rightMargin=1.75 * cm, leftMargin=1.75 * cm,
                                topMargin=2 * cm, bottomMargin=2 * cm)

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Normal_DejaVu', fontName='DejaVuSans', fontSize=10, leading=12))
        styles.add(ParagraphStyle(name='mHeading', fontName='DejaVuSans-Bold', fontSize=14, alignment=TA_CENTER, spaceAfter=12))
        styles.add(ParagraphStyle(name='SubHeading', fontName='DejaVuSans-Bold', fontSize=11, spaceAfter=10, textColor=colors.HexColor("#2E86C1")))

        elements = [
            Paragraph("Assignment Report", styles['mHeading']),
            Paragraph(f"Student Mail: <font color='black'>{student_name}</font>", styles['SubHeading']),
            Paragraph(f"Assignment Name: <font color='black'>{assignment_name}</font>", styles['SubHeading']),
            Spacer(1, 12)
        ]

        page_width, _ = A4
        available_width = page_width - doc.leftMargin - doc.rightMargin
        col_widths = [available_width * 0.35, available_width * 0.40, available_width * 0.15, available_width * 0.10]

        table_data = [
            [Paragraph(f"<b>{col}</b>", styles['Normal_DejaVu']) for col in ["Question", "Feedback", "Verdict", "Score"]]
        ]

        data["evaluation"] = self.sort_evaluation(data["evaluation"])

        for item in data["evaluation"]:
            for _, evaluation in item.items():
                question_key = next((k for k in evaluation if k.lower().startswith("question")), None)
                row = [
                    Paragraph(evaluation.get(question_key, ""), styles['Normal_DejaVu']),
                    Paragraph(evaluation.get("feedback", ""), styles['Normal_DejaVu']),
                    Paragraph(evaluation.get("verdict", ""), styles['Normal_DejaVu']),
                    Paragraph(str(evaluation.get("score", "")), styles['Normal_DejaVu']),
                ]
                table_data.append(row)

        table = Table(table_data, colWidths=col_widths, repeatRows=1)
        table.setStyle(TableStyle([
            ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
            ('BACKGROUND', (0, 0), (-1, 0), colors.orange),
            ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ]))

        elements.append(table)
        doc.build(elements, onFirstPage=self.decorator, onLaterPages=self.decorator)

    # ---------- upload PDF ----------
    def upload_pdf_to_s3(self, file_buffer, filename, metadata=None):
        content_type = "application/pdf"
        extra_args = {'ContentType': content_type}
        if metadata:
            extra_args['Metadata'] = {str(k): str(v) for k, v in metadata.items()}
        try:
            file_buffer.seek(0)
            self.S3_CLIENT.upload_fileobj(
                file_buffer,
                self.S3_BUCKET_NAME,
                filename,
                ExtraArgs=extra_args
            )
            return f"https://{self.S3_BUCKET_NAME}.s3.amazonaws.com/{filename}"
        except Exception as e:
            return {"error": "S3 upload failed", "details": str(e)}

    # ---------- DynamoDB student lookup ----------
    def get_student_data(self, email):
        if not email:
            return None
        try:
            resp = self.STUDENTS_TABLE.get_item(Key={"email": email})
            return resp.get("Item")
        except Exception as e:
            print(f"Failed to fetch student data for {email}: {e}")
            return None

    # ---------- Get assignment metadata (from generated-assignments bucket) ----------
    def get_assignment_metadata(self, assignment_name):
        try:
            resp = self.S3_CLIENT.head_object(Bucket=self.GENERATED_ASSIGNMENTS_BUCKET, Key=assignment_name)
            metadata = resp.get("Metadata", {})
            # metadata keys are lowercase in S3
            return {
                "class_code": metadata.get("class_code"),
                "deadline": metadata.get("deadline"),
                "extra_metadata": metadata
            }
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            print(f"Error fetching assignment metadata ({assignment_name}): {code} - {e}")
            return None
        except Exception as e:
            print(f"Unexpected error fetching assignment metadata: {e}")
            return None

    # ---------- parse ISO datetime to UTC-aware datetime for comparison ----------
    @staticmethod
    def parse_iso_datetime_to_utc(dt_str):
        """
        Accepts ISO datetime string. Returns naive UTC datetime for comparison with datetime.datetime.utcnow()
        If dt_str contains timezone, convert to UTC and return naive UTC.
        """
        if not dt_str:
            return None
        try:
            # Python 3.11+ fromisoformat supports offset; for older versions this still fairly robust
            dt = datetime.datetime.fromisoformat(dt_str)
        except Exception:
            # Fall back to common formats
            try:
                dt = datetime.datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S")
            except Exception:
                try:
                    dt = datetime.datetime.strptime(dt_str, "%Y-%m-%d")
                    # set to end of day
                    dt = dt.replace(hour=23, minute=59, second=59)
                except Exception:
                    return None

        # If timezone-aware, convert to UTC and then drop tzinfo
        if dt.tzinfo is not None:
            dt_utc = dt.astimezone(datetime.timezone.utc).replace(tzinfo=None)
            return dt_utc
        # naive => assume UTC
        return dt
    
    def send_student_email(self,to_email, subject, body):
        smtp_server = os.getenv("SMTP_SERVER")
        smtp_port = int(os.getenv("SMTP_PORT"))
        smtp_user = os.getenv("SMTP_USER")
        smtp_pass = os.getenv("SMTP_PASS")
        from_email = os.getenv("FROM_EMAIL")

        msg = MIMEMultipart()
        msg["From"] = from_email
        msg["To"] = to_email
        msg["Subject"] = subject

        msg.attach(MIMEText(body, "plain"))

        try:
            with smtplib.SMTP(smtp_server, smtp_port) as server:
                server.starttls()
                server.login(smtp_user, smtp_pass)
                server.sendmail(from_email, to_email, msg.as_string())
            print(f"📧 Mail sent successfully to {to_email}")
        except Exception as e:
            print(f"❌ Failed to send email: {e}")

    # ---------- Excel update/creation ----------
    def update_assignment_excel(self, assignment_name, student_data, evaluation, deadline, bucket=None):
        bucket = bucket or self.EXCEL_BUCKET

        # Build filename: college_classcode_assignment.xlsx
        college = (student_data.get("college_name") if student_data else "unknownCollege") or "unknownCollege"
        college_safe = college.replace(" ", "_")
        class_code = (student_data.get("class_code") if student_data else "unknownClass") or "unknownClass"
        safe_assignment = assignment_name.replace(" ", "_")
        key = f"{college_safe}_{class_code}_{safe_assignment}.xlsx"

        # Parse deadline to compare
        if isinstance(deadline, str):
            parsed_deadline = self.parse_iso_datetime_to_utc(deadline)
        elif isinstance(deadline, datetime.datetime):
            parsed_deadline = deadline
        else:
            parsed_deadline = None

        if parsed_deadline is None:
            return {"error": "Invalid or missing deadline metadata. Excel not updated.", "key": key}

        now_utc = datetime.datetime.utcnow()
        if now_utc > parsed_deadline:
            return {"error": f"Deadline passed for {assignment_name}. Excel is locked.", "key": key}

        # Try to load existing workbook from S3
        wb = None
        ws = None
        try:
            obj = self.S3_CLIENT.get_object(Bucket=bucket, Key=key)
            file_bytes = io.BytesIO(obj["Body"].read())
            wb = load_workbook(file_bytes)
            ws = wb.active
        except ClientError as e:
            err_code = e.response.get("Error", {}).get("Code", "")
            # If not found, create new workbook
            if err_code in ("NoSuchKey", "404", "NotFound", "NoSuchBucket"):
                wb = Workbook()
                ws = wb.active
                ws.title = "Submissions"
                ws.append([
                    "Name", "Email", "Phone", "College", "Class Code", "Department",
                    "Question", "Feedback", "Verdict", "Score"
                ])
            else:
                print(f"S3 get_object failed for {key}: {e}")
                return {"error": "Failed to load existing Excel", "details": str(e)}
        except Exception as e:
            # Some other error (e.g., bucket doesn't exist) -> create workbook if we choose to
            print(f"Unexpected error loading workbook from S3: {e}")
            wb = Workbook()
            ws = wb.active
            ws.title = "Submissions"
            ws.append([
                "Name", "Email", "Phone", "College", "Class Code", "Department",
                "Question", "Feedback", "Verdict", "Score"
            ])

        # Append evaluation rows: one row per question
        try:
            eval_list = evaluation.get("evaluation", [])
            for item in eval_list:
                # item is like {"Q1": { ... }}
                for _, qdata in item.items():
                    question_key = next((k for k in qdata if k.lower().startswith("question")), None)
                    ws.append([
                        student_data.get("name", "") if student_data else "",
                        student_data.get("email", "") if student_data else "",
                        student_data.get("phone", "") if student_data else "",
                        student_data.get("college_name", "") if student_data else "",
                        student_data.get("class_code", "") if student_data else "",
                        student_data.get("department", "") if student_data else "",
                        qdata.get(question_key, "") if question_key else "",
                        qdata.get("feedback", ""),
                        qdata.get("verdict", ""),
                        qdata.get("score", "")
                    ])
        except Exception as e:
            print(f"Failed to append evaluation rows: {e}")
            return {"error": "Appending rows failed", "details": str(e)}

        # Save workbook to buffer and upload back to S3
        try:
            buffer = io.BytesIO()
            wb.save(buffer)
            buffer.seek(0)
            self.S3_CLIENT.upload_fileobj(
                buffer,
                bucket,
                key,
                ExtraArgs={"ContentType": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"}
            )
            file_url = f"https://{bucket}.s3.amazonaws.com/{key}"
            return {"message": "Excel updated successfully", "fileUrl": file_url, "key": key}
        except Exception as e:
            print(f"Failed to upload Excel to S3: {e}")
            return {"error": "Excel upload failed", "details": str(e)}

# ---------- Evaluator (Lambda-like handler) ----------
class Evaluator:
    def evl(self, event):
        obj = Processor()
        body = event['request']['body']
        if isinstance(body, str):
            body = json.loads(body)

        # presigned or direct file URL posted in body
        presigned_url = body.get("file_path")
        if not presigned_url:
            return {"statusCode": 400, "body": json.dumps({"error": "No file_path provided in request body"})}

        print("📥 Downloading file from S3...")
        try:
            resp = requests.get(presigned_url)
            resp.raise_for_status()
        except Exception as e:
            print(f"Failed to download file from presigned URL: {e}")
            return {"statusCode": 400, "body": json.dumps({"error": "Failed to download file", "details": str(e)})}

        file_bytes = io.BytesIO(resp.content)
        print("📄 Extracting text from PDF...")
        try:
            extracted_text = obj.read_pdf(file_bytes)
        except Exception as e:
            print(f"PDF reading failed: {e}")
            return {"statusCode": 500, "body": json.dumps({"error": "Failed to extract text from PDF", "details": str(e)})}

        print("🔍 Chunking text for evaluation...")
        chunks = obj.chunk_text(extracted_text)
        evaluations = []

        print("🧠 Running GPT evaluations...")
        for idx, chunk in enumerate(chunks):
            print(f"→ Evaluating chunk {idx+1}/{len(chunks)}...")
            chunk_output = obj.evaluate_entire_assignment(chunk)
            cleaned = obj.cleaner({"evaluation": chunk_output})
            # if cleaner returns error, stop and return
            if "error" in cleaned:
                print(f"Cleaner failed for chunk {idx+1}: {cleaned}")
                return {"statusCode": 500, "body": json.dumps({"error": "Cleaner failed", "details": cleaned})}
            evaluations.extend(cleaned.get("evaluation", []))

        cleaned_data = {"evaluation": evaluations}

        print("✅ Verifying output...")
        try:
            verified, reason = obj.verify_evaluation(json.dumps(cleaned_data))
        except Exception as e:
            print(f"Verification call failed: {e}")
            return {"statusCode": 500, "body": json.dumps({"error": "Verification failed", "details": str(e)})}

        if not verified:
            print(f"❌ JSON verification failed: {reason}")
            return {"statusCode": 400, "body": json.dumps({"error": reason})}

        print("📝 Generating PDF...")
        # output_pdf_path = "evaluation_report.pdf"
        try:
            obj.json_to_pdf_report(cleaned_data, body.get("student_name", ""), body.get("assignment_name", ""), output_pdf_path)
        except Exception as e:
            print(f"PDF generation failed: {e}")
            return {"statusCode": 500, "body": json.dumps({"error": "PDF generation failed", "details": str(e)})}

        print("☁️ Uploading PDF to S3...")
        try:
            with open(output_pdf_path, "rb") as f:
                buffer = io.BytesIO(f.read())
                std = body.get("student_name", "").split('@')[0]
                assn = body.get("assignment_name", "").split('.')[0]
                pdf_filename = f"{std}_{assn}.pdf"
                pdf_url = obj.upload_pdf_to_s3(
                    buffer,
                    filename=pdf_filename,
                    metadata={
                        "email": body.get("student_name"),
                        "assignment_name": body.get("assignment_name"),
                        "class_code": body.get("class_code", "")
                    }
                )
        except Exception as e:
            print(f"S3 PDF upload failed: {e}")
            return {"statusCode": 500, "body": json.dumps({"error": "PDF upload failed", "details": str(e)})}
        
        s3_client = boto3.client("s3", region_name="us-east-1")
        presigned_url = s3_client.generate_presigned_url(
            "get_object",
            Params={
                "Bucket": obj.S3_BUCKET_NAME,
                "Key": f"{std}_{assn}.pdf"
            },
            ExpiresIn=604800  # 7 days
        )

        # Send mail to student
        self.send_student_email(
            to_email=body.get("student_name"),  # student's email
            subject=f"Your Evaluation Report for {body.get('assignment_name')}",
            body=f"""Hello,

        Your assignment has been evaluated.

        You can download your report from the link below (valid for 7 days):
        {presigned_url}

        Best regards,
        Assignment Evaluator
        """
        )


        # ---------- NEW: Excel update ----------
        print("📥 Fetching assignment metadata (deadline & class_code)...")
        assignment_name = body.get("assignment_name")
        if not assignment_name:
            return {"statusCode": 400, "body": json.dumps({"error": "assignment_name missing in request body"})}

        assignment_meta = obj.get_assignment_metadata(assignment_name)
        if not assignment_meta:
            # if metadata fetch failed, still return success for evaluation but note metadata fetch failure
            return {
                "statusCode": 200,
                "headers": {
                    "Access-Control-Allow-Origin": "*",
                    "Access-Control-Allow-Methods": "OPTIONS,POST,GET",
                    "Access-Control-Allow-Headers": "Content-Type"
                },
                "body": json.dumps({
                    "message": "Evaluation completed and PDF uploaded, but failed to fetch assignment metadata for Excel update.",
                    "fileUrl": pdf_url
                })
            }

        deadline_str = assignment_meta.get("deadline")
        # parse deadline
        parsed_deadline = obj.parse_iso_datetime_to_utc(deadline_str)
        if parsed_deadline is None:
            # same fallback: return success for evaluation but warn about deadline parse issue
            return {
                "statusCode": 200,
                "headers": {
                    "Access-Control-Allow-Origin": "*",
                    "Access-Control-Allow-Methods": "OPTIONS,POST,GET",
                    "Access-Control-Allow-Headers": "Content-Type"
                },
                "body": json.dumps({
                    "message": "Evaluation completed and PDF uploaded, but assignment deadline metadata was invalid. Excel not updated.",
                    "fileUrl": pdf_url
                })
            }

        # Fetch student data from DynamoDB using the email (student_name field contains email)
        student_email = body.get("student_name")  # you use student_name as email in previous code
        student_data = obj.get_student_data(student_email)
        if not student_data:
            # If no student data, still return success of PDF but report missing student entry
            return {
                "statusCode": 200,
                "headers": {
                    "Access-Control-Allow-Origin": "*",
                    "Access-Control-Allow-Methods": "OPTIONS,POST,GET",
                    "Access-Control-Allow-Headers": "Content-Type"
                },
                "body": json.dumps({
                    "message": "Evaluation completed and PDF uploaded, but student record not found in DynamoDB. Excel not updated.",
                    "fileUrl": pdf_url
                })
            }

        # Update excel
        print("📊 Updating Excel tracker in S3...")
        try:
            excel_result = obj.update_assignment_excel(
                assignment_name=assignment_name,
                student_data=student_data,
                evaluation=cleaned_data,
                deadline=parsed_deadline,
                bucket=obj.EXCEL_BUCKET
            )
        except Exception as e:
            print(f"Excel update failed unexpectedly: {e}")
            excel_result = {"error": "Excel update threw exception", "details": str(e)}

        # Build final response: include pdf_url and excel_result (if any)
        response_body = {
            "message": "Evaluation report generated and uploaded successfully.",
            "fileUrl": pdf_url,
            "excel_result": excel_result
        }

        return {
            "statusCode": 200,
            "headers": {
                "Access-Control-Allow-Origin": "*",
                "Access-Control-Allow-Methods": "OPTIONS,POST,GET",
                "Access-Control-Allow-Headers": "Content-Type"
            },
            "body": json.dumps(response_body)
        }


In [15]:
test_event = {
    "request": {
        "body": json.dumps({
            "file_path": "https://submitted-assignments.s3.us-east-1.amazonaws.com/assn_sub.pdf",
            "student_name": "bilal@inikola.com",
            "assignment_name": "python_quiz_assignment.pdf",
            "batch_name": "GenAIDSI-2"
        })
    }
}


In [16]:
test_event

{'request': {'body': '{"file_path": "https://submitted-assignments.s3.us-east-1.amazonaws.com/assn_sub.pdf", "student_name": "bilal@inikola.com", "assignment_name": "python_quiz_assignment.pdf", "batch_name": "GenAIDSI-2"}'}}

In [17]:
test_event = {
    "request": {
        "body": json.dumps({
            "file_path": "https://submitted-assignments.s3.us-east-1.amazonaws.com/assn_sub.pdf",
            "student_name": "bilal@inikola.com",
            "assignment_name": "python_quiz_assignment.pdf",
            "class_code": "CS101"
        })
    }
}


In [19]:
evaluator = Evaluator()
response = evaluator.evl(test_event)
print(json.dumps(response, indent=2))

📥 Downloading file from S3...
Failed to download file from presigned URL: 403 Client Error: Forbidden for url: https://submitted-assignments.s3.us-east-1.amazonaws.com/assn_sub.pdf
{
  "statusCode": 400,
  "body": "{\"error\": \"Failed to download file\", \"details\": \"403 Client Error: Forbidden for url: https://submitted-assignments.s3.us-east-1.amazonaws.com/assn_sub.pdf\"}"
}
